# Hong Kong Air Quality Briefing System
## Task D · Explore 2025 observations

**User inputs → SQLite saved queries → Template rendering → Visualization → Static export**

Open this notebook from the project root. Install the dependencies in the kernel environment with `%pip install -r requirements_task_d.txt` if needed. In Colab, upload or mount the project and use `%cd` to enter its root before installing dependencies and running the cells. You can also set `AQHI_PROJECT_ROOT` to the project folder.

Reports and writes use a temporary SQLite snapshot. The export cell produces a temporary website that can be rebuilt to `docs/` from the project root.

In [ ]:
import os
import sys
from contextlib import ExitStack
from pathlib import Path
from tempfile import TemporaryDirectory
import atexit

configured = os.environ.get("AQHI_PROJECT_ROOT")
candidates = [Path(configured)] if configured else [Path.cwd(), *Path.cwd().parents]
project = next((p.resolve() for p in candidates if (p / "src/schema.py").is_file()
                and (p / "data/db/air_quality.db").is_file()), None)
if project is None:
    raise FileNotFoundError("Enter the project root or set AQHI_PROJECT_ROOT.")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

import pandas as pd
from IPython.display import HTML, Image, display
from src.schema import DEFAULT_DB_PATH, PROJECT_ROOT, get_connection
from src.site_export import export_static_site, validate_site_links
from src.task_d_integration import default_cases, demo_database, make_briefing, query_catalog, select
from src.viz import build_district_map, plot_category_distribution, plot_hourly_aqhi, plot_monthly_trend, plot_station_comparison

assert PROJECT_ROOT == project, f"The imported src.schema is from {PROJECT_ROOT}"
counts = select("""
    SELECT (SELECT COUNT(*) FROM districts) AS districts,
           (SELECT COUNT(*) FROM stations) AS stations,
           (SELECT COUNT(*) FROM hourly_aqhi) AS aqhi_rows,
           (SELECT COUNT(*) FROM hourly_pollutants) AS pollutant_rows,
           (SELECT COUNT(*) FROM daily_weather) AS weather_days
""")[0]
assert all(counts.values()), counts
display(pd.DataFrame([counts]))

if "task_d_resources" in globals():
    task_d_resources.close()
task_d_resources = ExitStack()
atexit.register(task_d_resources.close)
demo_db = task_d_resources.enter_context(demo_database())
work = Path(task_d_resources.enter_context(TemporaryDirectory(prefix="aqhi-notebook-")))
stations, dates, months = query_catalog()
station_names = [row["name"] for row in stations]
template_rows = select("""
    SELECT t.name, q.query_code, q.parameter_names FROM report_templates t
    JOIN saved_queries q USING(query_id) ORDER BY q.query_code
""", db_path=demo_db)
template_names = [row["name"] for row in template_rows]
assert len(template_names) >= 8
cases = default_cases()
first = cases[0]
print(f"{len(station_names)} stations, {dates[0]} through {dates[-1]}, {len(template_names)} report types")
display(pd.DataFrame(template_rows)[["name", "query_code"]])

## Build a briefing, charts or district map

Select a report type and its inputs. **Generate briefing** uses the report's Date, Month or Range fields as appropriate. **Draw four charts** uses Date for the three daily charts and Month for the trend. **Show District Map** uses the current Date selection for any day in 2025.

If the widget controls do not render, edit `interactive_inputs` in the parameter cell below and rerun it. Edit `day` to generate the daily briefing, charts and map for that date.

In [ ]:
import json
from IPython.display import clear_output

query_params = {r["name"]: set(json.loads(r["parameter_names"])) for r in template_rows}
preferred = ["station_name", "station_type", "district_name", "row_type", "obs_date",
             "month", "category", "hours", "percentage", "valid_hours", "missing_hours",
             "substitute_hours", "min_aqhi", "max_aqhi", "peak_hours", "high_or_above_hours",
             "avg_pm25", "valid_pm25_hours", "high_risk_hours", "peak_stations"]

def show_briefing(template, station, other, day, month, start, end):
    params = dict(station_name=station, station_a=station, station_b=other,
                  obs_date=day, month=month, start_date=start, end_date=end)
    result = make_briefing(template, params, demo_db)
    print(f"{result.query_code} · {len(result.rows)} SQL result rows · "
          + ("formal template" if result.formal else "query-data fallback"))
    frame = pd.DataFrame(result.rows)
    if not frame.empty:
        display(frame[[name for name in preferred if name in frame.columns]].head(24))
    display(HTML(result.outputs["html_output"]))
    return result

def show_charts(station, other, day, month):
    for index, (plot, args) in enumerate((
        (plot_hourly_aqhi, (station, day)),
        (plot_category_distribution, (station, day)),
        (plot_station_comparison, (station, other, day)),
        (plot_monthly_trend, (station, month)),
    )):
        path = plot(*args, work / f"chart-{index}.png", db_path=demo_db)
        display(Image(filename=str(path), width=950))

def show_district_map(day):
    district_map = build_district_map(day, work / f"districts-{day}.html", db_path=demo_db)
    print(day, district_map.task_d_query_source)
    display(district_map)
    return district_map

try:
    if os.environ.get("AQHI_DISABLE_WIDGETS") == "1":
        raise ImportError("Widgets disabled")
    import ipywidgets as widgets
    if "google.colab" in sys.modules:
        from google.colab import output
        output.enable_custom_widget_manager()
except ImportError:
    widgets = None
    print("Use the editable interactive_inputs cell below.")

if widgets is not None:
    names = [(f'{r["name"]} ({r["station_type"]})', r["name"]) for r in stations]
    controls = {
        "template": widgets.Dropdown(options=template_names, description="Report"),
        "station": widgets.Dropdown(options=names, value=first["station_name"], description="Station"),
        "other": widgets.Dropdown(options=names, value=first["station_b"], description="Second"),
        "day": widgets.Dropdown(options=dates, value=first["obs_date"], description="Date"),
        "month": widgets.Dropdown(options=months, value=first["month"], description="Month"),
        "start": widgets.Dropdown(options=dates, value=first["obs_date"], description="Range start"),
        "end": widgets.Dropdown(options=dates, value=first["obs_date"], description="Range end"),
    }
    for control in controls.values():
        control.layout.width = "min(100%, 480px)"
        control.style.description_width = "105px"
    briefing_button = widgets.Button(description="Generate briefing", button_style="primary")
    chart_button = widgets.Button(description="Draw four charts")
    map_button = widgets.Button(description="Show District Map", layout=widgets.Layout(width="auto"))
    result_area = widgets.Output()

    def update_report_inputs(change=None):
        required = query_params[controls["template"].value]
        for key, param in (("start", "start_date"), ("end", "end_date")):
            controls[key].layout.display = "" if param in required else "none"
        briefing_button.tooltip = "Report fields: " + ", ".join(sorted(required))

    def on_click(button):
        values = {key: control.value for key, control in controls.items()}
        with result_area:
            clear_output(wait=True)
            try:
                if button is briefing_button:
                    show_briefing(**values)
                elif button is map_button:
                    show_district_map(values["day"])
                else:
                    show_charts(*(values[key] for key in ("station", "other", "day", "month")))
            except ValueError as exc:
                print(exc)

    controls["template"].observe(update_report_inputs, names="value")
    update_report_inputs()
    buttons = (briefing_button, chart_button, map_button)
    for button in buttons:
        button.on_click(on_click)
    display(widgets.VBox([
        *controls.values(),
        widgets.HBox(buttons, layout=widgets.Layout(flex_flow="row wrap")),
        result_area,
    ]))

In [ ]:
# Edit these values to run with or without widget support.
interactive_inputs = dict(
    template=template_names[0], station=first["station_name"], other=first["station_b"],
    day=first["obs_date"], month=first["month"], start=first["obs_date"], end=first["obs_date"],
)
briefing = show_briefing(**interactive_inputs)
show_charts(*(interactive_inputs[k] for k in ("station", "other", "day", "month")))
selected_map = show_district_map(interactive_inputs["day"])

## Optional district map examples

Uncomment the last line below and run the cell to compare the default demonstration dates, 1 April 2025 and 14 April 2025. The interactive map above uses the Date you select.

District fill uses the maximum valid AQHI among General stations in each district. Toggle General and Roadside separately and click a station for observations and population context. Folium loads Leaflet and basemap resources in your browser.

In [ ]:
def show_example_maps():
    for case in cases:
        show_district_map(case["obs_date"])

# show_example_maps()

## Export and verify

The export includes the charts, district maps and eight sample briefings when the source has empty report history. The exporter verifies all relative HTML references. To write the publication directory from the project root, run `python -m src.site_export`.

In [ ]:
index = export_static_site(output_dir=work / "site")
errors = validate_site_links(index.parent)
assert index.is_file() and not errors, errors
print("Verified static index", index)
print("Report HTML files", len(list((index.parent / "reports").glob("*.html"))))
print("Map HTML files", len(list((index.parent / "maps").glob("*.html"))))